# Lesson 11 - Model Context Protocol (MCP)

Ang **Model Context Protocol (MCP)** ay isang bukas na pamantayan na nagpapahintulot sa mga ahente na dinamiko na tuklasin at gamitin ang mga kasangkapan, mapagkukunan, at pinagkukunan ng datos sa takbo ng oras. Sa halip na ipako ang mga kasangkapan sa isang ahente, pinapayagan ng MCP ang mga ahente na kumonekta sa mga panlabas na server na naglalantad ng mga kakayahan kapag kinakailangan.

Sa araling ito, matututuhan mo:
- Ano ang MCP at bakit ito mahalaga para sa mga sistema ng ahente
- Paano gumagana ang arkitekturang kliyente-server ng MCP
- Paano bumuo ng mga ahenteng gumagamit ng pagtuklas ng kasangkapan sa istilong MCP


## Setup

**Mga Kinakailangan:**
- Microsoft Foundry project na may naka-deploy na modelo
- Patakbuhin ang `az login` para sa pagpapatunay


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Ano ang Model Context Protocol (MCP)?

Ang MCP ay nagtatakda ng isang pamantayang paraan para sa mga AI agent na matuklasan at makipag-ugnayan sa mga panlabas na kasangkapan at mga pinagkukunan ng datos:

- **MCP Server**: Nagpapakita ng mga kasangkapan, mga yaman, at mga prompt sa pamamagitan ng isang pamantayang protocol
- **MCP Client**: Ang runtime ng agent na kumokonekta sa mga server at natutuklasan ang magagamit na mga kakayahan
- **Dynamic Discovery**: Hindi kailangan ng mga agent ng hardcoded na mga kasangkapan — natutuklasan nila ang magagamit sa runtime

Malakas ito para sa pagbuo ng mga extensible na sistema ng agent kung saan maaaring idagdag ang mga bagong kakayahan nang hindi binabago ang code ng agent.


## Paano Gumagana ang MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Ang ahente (kliyente ng MCP) ay kumokonekta sa isang server ng MCP
2. Tumatanggap ang server ng listahan ng mga magagamit na kasangkapan at ang kanilang mga iskema
3. Maaari nang tawagin ng ahente ang anumang natuklasang kasangkapan habang nagrereason
4. Ang mga resulta ay nagbabalik sa parehong protocol


## Pagsasagawa ng Simulasyon sa Pagdiskubre ng MCP Tool

Dahil ang tunay na MCP server ay nangangailangan ng gumaganang server process, ipapakita namin ang pattern gamit ang mga `@tool` functions na nagsisimula kung ano ang maibibigay ng isang MCP-connected accommodation service.

Sa produksyon, ang mga tool na ito ay madidiskubre nang dinamiko mula sa isang MCP server sa halip na maidetalye nang lokal.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Paggawa ng Agent gamit ang MCP-Style Tools


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP sa Produksyon

Sa isang production environment, pinapayagan ng MCP ang mga makapangyarihang pattern:

- **Dynamic tool discovery**: Kumokonekta ang mga ahente sa MCP server at natutuklasan ang mga tool habang tumatakbo
- **Decoupled architecture**: Ang mga tagapagbigay ng tool ay maaaring mag-update nang independent mula sa ahente
- **Cross-organization sharing**: Maaaring ipakita ng mga team ang mga kakayahan sa pamamagitan ng MCP server na magagamit ng kahit anong ahente
- **Microsoft Agent Framework support**: May built-in MCP client support ang MAF sa pamamagitan ng `mcp` integration

Upang gumamit ng totoong MCP server kasama ang MAF, kumonekta ka gamit ang `hosted_mcp_tool()` o ang MCP client integration.

**Alamin pa:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Buod

Sa araling ito, natutunan mo:
- Ang **MCP** ay isang bukas na pamantayan para sa dynamic na pagtuklas ng mga tool sa pagitan ng mga ahente at mga tagapagbigay ng tool
- Ang **client-server architecture** ay nagpapahintulot sa mga ahente na matuklasan ang mga kakayahan sa panahon ng pagtakbo
- Pinapagana ng MCP ang **mapapalawak, hindi magkakabit na mga sistema ng ahente** kung saan maaaring idagdag ang mga tool nang walang pagbabago sa code
- Nagbibigay ang Microsoft Agent Framework ng **built-in na suporta sa MCP** para sa produksyon


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
